<a href="https://colab.research.google.com/github/pxtroniwnl/barcelona-de-indias-time-serie/blob/main/temperaturaIDEAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Temperatura del aire — estaciones IDEAM cercanas al ROI de la laguna

**Fuente:** `Temperatura_Ambiente_del_Aire_20260818_SOLO_BOLIVAR(in).csv` (IDEAM, departamento de Bolívar)

**Objetivo:** identificar y caracterizar las estaciones meteorológicas más cercanas al área de estudio
(laguna peri-urbana en el norte de Cartagena) y extraer su serie de temperatura del aire a 2 m.

## Estructura

| Sección | Contenido |
|---|---|
| 1 | Configuración, constantes y autenticación GEE |
| 2 | Carga del CSV crudo (`df_raw`, inmutable) |
| 3 | Limpieza y normalización (`df`) |
| 4 | Catálogo de estaciones |
| 5 | Geometría: ROI y cajas anidadas |
| 6 | Clasificación espacial y distancias |
| 7 | Mapa |
| 8 | Selección de estaciones y serie temporal |
| 9 | Diagnóstico de cobertura y frecuencia |
| 10 | Limitaciones |

## Principio de organización

Cada celda es **idempotente**: puede reejecutarse sin corromper el estado. El crudo (`df_raw`)
nunca se sobrescribe; toda transformación usa `.copy()` y devuelve un objeto nuevo. Esto permite
`Runtime > Run all` sin sorpresas.

## 1. Configuración y autenticación

In [ ]:
# Instalación (solo en Colab, la primera vez)
# !pip install -q pandas geemap earthengine-api

import csv
import io
import math
from pathlib import Path

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------- rutas
RUTA_CSV = Path("/content/Temperatura_Ambiente_del_Aire_20260818_SOLO_BOLIVAR(in).csv")

# ---------------------------------------------------------------- parseo
# El CSV viene doblemente entrecomillado: cada fila completa es un solo campo.
FORMATO_FECHA = "%Y %b %d %I:%M:%S %p"  # ej. "2020 Jan 21 09:00:00 PM"

# Columnas que deben conservarse como texto (los ceros a la izquierda son parte del ID)
DTYPES_TEXTO = {"CodigoEstacion": "string", "CodigoSensor": "string"}

COLS_TEXTO = [
    "CodigoEstacion", "CodigoSensor", "NombreEstacion", "Departamento",
    "Municipio", "ZonaHidrografica", "DescripcionSensor", "UnidadMedida",
]

# ---------------------------------------------------------------- control de calidad
# Rango físicamente plausible para temperatura del aire en el Caribe colombiano.
# Fuera de este rango se marca como NaN (captura centinelas tipo -9999).
RANGO_TEMP_VALIDO = (5.0, 50.0)

# ---------------------------------------------------------------- geometría
# Vértices del ROI (laguna). Longitud primero, luego latitud.
ROI_COORDS = [
    [-75.476052, 10.517524],
    [-75.476117, 10.518747],
    [-75.473158, 10.519223],
    [-75.470516, 10.525108],
    [-75.469572, 10.524876],
    [-75.471686, 10.518916],
    [-75.468394, 10.517219],
    [-75.468952, 10.516459],
]

# Factores de escalado de las cajas anidadas.
# NOTA: el factor multiplica el LADO, no el área. Una caja 2x tiene 4x el área.
FACTORES = [1, 2, 3, 4]
COLORES_CAJA = {1: "FF0000", 2: "FF8800", 3: "FFFF00", 4: "00FF00"}

# Número de estaciones a seleccionar para el análisis
N_ESTACIONES = 2

# Tolerancia para considerar dos registros el mismo sitio físico (grados ≈ 110 m)
TOL_SITIO = 3  # decimales de redondeo

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
# sin float_format global: forzaría los mismos decimales a coordenadas y a temperaturas

# ---------------------------------------------------------------- Earth Engine
PROYECTO_GEE = "proyecto1-504013"

ee.Authenticate(auth_mode="notebook")
ee.Initialize(project=PROYECTO_GEE)
print("GEE listo:", ee.String("ok").getInfo())

## 2. Carga del CSV crudo

El archivo tiene una capa extra de comillas: cada línea completa está envuelta como un único campo
y las comillas internas vienen duplicadas. Se desescapa con el módulo `csv` antes de pasar a pandas.

`df_raw` **no se modifica nunca**. Para volver a un estado limpio basta reejecutar esta celda.

In [ ]:
def cargar_crudo(ruta: Path) -> pd.DataFrame:
    """Desescapa el doble entrecomillado del CSV del IDEAM y lo carga en pandas."""
    with open(ruta, encoding="utf-8") as f:
        lineas = [fila[0] for fila in csv.reader(f) if fila]

    return pd.read_csv(io.StringIO("\n".join(lineas)), dtype=DTYPES_TEXTO)


df_raw = cargar_crudo(RUTA_CSV)
print(f"{len(df_raw):,} filas × {df_raw.shape[1]} columnas")
df_raw.head(3)

In [ ]:
# Diagnóstico del crudo: valores categóricos y encoding
for col in ["UnidadMedida", "DescripcionSensor", "CodigoSensor", "Departamento"]:
    vals = sorted(df_raw[col].dropna().unique().tolist())
    print(f"{col:20s} ({len(vals):>3}): {vals[:6]}{' …' if len(vals) > 6 else ''}")

print()
print(df_raw.dtypes)

## 3. Limpieza

Cuatro operaciones, en este orden:

1. **Normalización de texto** — `.str.strip()` para los extremos y colapso de espacios internos
   múltiples (`GALERAZAMBA  - AUT` → `GALERAZAMBA - AUT`). Sin esto, agrupar por nombre produce
   categorías espurias.
2. **Parseo de fecha** — después de limpiar, para que espacios sobrantes no rompan el parseo.
   Se pasa `format=` explícito: sin él pandas infiere fila por fila y es órdenes de magnitud más lento.
3. **Control de rango** — valores fuera de `RANGO_TEMP_VALIDO` pasan a `NaN` en vez de eliminarse,
   preservando la marca temporal del registro fallido.
4. **Desduplicación** — por `(CodigoEstacion, FechaObservacion)`, **no** incluyendo `CodigoSensor`.

El punto 4 es una corrección importante. El dataset trae varios códigos de sensor para la misma
variable física (`Temp Aire 2 m`, `TEMPERATURA DEL AIRE A 2 m`, `GPRS - TEMPERATURA DEL AIRE A 2 m`).
Incluir `CodigoSensor` en la clave de desduplicación deja pasar registros repetidos del mismo
instante medidos por el mismo instrumento bajo dos etiquetas distintas.

In [ ]:
def limpiar(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Normaliza texto, parsea fechas, filtra valores implausibles y desduplica."""
    df = df_raw.copy()

    # 1. texto: extremos + espacios internos colapsados
    for c in COLS_TEXTO:
        if c in df.columns:
            df[c] = df[c].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

    # 2. fecha
    df["FechaObservacion"] = pd.to_datetime(df["FechaObservacion"], format=FORMATO_FECHA)

    # 3. rango físico → NaN (no se borra la fila)
    lo, hi = RANGO_TEMP_VALIDO
    df["ValorObservado"] = pd.to_numeric(df["ValorObservado"], errors="coerce")
    df.loc[~df["ValorObservado"].between(lo, hi), "ValorObservado"] = np.nan

    # 4. desduplicación determinista: se ordena antes para que keep="first" sea reproducible
    df = (
        df.sort_values(["CodigoEstacion", "FechaObservacion", "CodigoSensor"], kind="stable")
        .drop_duplicates(subset=["CodigoEstacion", "FechaObservacion"], keep="first")
        .reset_index(drop=True)
    )
    return df


df = limpiar(df_raw)

print(f"crudo   : {len(df_raw):,}")
print(f"limpio  : {len(df):,}  ({len(df_raw) - len(df):,} duplicados eliminados)")
print(f"NaN QC  : {df['ValorObservado'].isna().sum():,} valores fuera de {RANGO_TEMP_VALIDO}")
df.dtypes

## 4. Catálogo de estaciones

Un mismo `CodigoEstacion` aparece con **varias grafías de nombre** en el archivo
(`AEROPUERTO RAFAEL NUNEZ` vs `AEROPUERTO RAFAEL NÚÑEZ - AUT [14015080]`). Agrupar por
`(código, nombre)` fragmenta la misma estación en varias filas — es lo que producía cinco
registros para dos estaciones en la versión anterior.

Aquí se elige un **nombre canónico** por código: el más frecuente, con desempate alfabético
para que el resultado sea determinista.

In [ ]:
def construir_catalogo(df: pd.DataFrame) -> pd.DataFrame:
    """Una fila por CodigoEstacion, con nombre canónico y coordenadas robustas."""
    # nombre más frecuente por código; desempate alfabético para reproducibilidad
    conteo = (
        df.groupby(["CodigoEstacion", "NombreEstacion"], as_index=False)
        .size()
        .rename(columns={"size": "_n"})
    )
    nombre_canonico = (
        conteo.sort_values(
            ["CodigoEstacion", "_n", "NombreEstacion"],
            ascending=[True, False, True],
            kind="stable",
        )
        .drop_duplicates("CodigoEstacion", keep="first")
        .drop(columns="_n")
    )

    agregados = (
        df.groupby("CodigoEstacion")
        .agg(
            Municipio=("Municipio", lambda s: s.mode().iat[0]),
            ZonaHidrografica=("ZonaHidrografica", lambda s: s.mode().iat[0]),
            # mediana: robusta ante coordenadas truncadas en algunos registros
            Latitud=("Latitud", "median"),
            Longitud=("Longitud", "median"),
            n_variantes_nombre=("NombreEstacion", "nunique"),
            n_sensores=("CodigoSensor", "nunique"),
            n_obs=("ValorObservado", "size"),
        )
        .reset_index()
    )

    return nombre_canonico.merge(agregados, on="CodigoEstacion", how="left")


catalogo = construir_catalogo(df)
print(f"{len(catalogo)} estaciones únicas")
catalogo.sort_values("n_obs", ascending=False)

## 5. Geometría: ROI y cajas anidadas

In [ ]:
roi = ee.Geometry.Polygon([ROI_COORDS + [ROI_COORDS[0]]])  # cierre explícito del polígono

# Centro y semiejes del bounding box del ROI (cálculo local, sin llamadas a GEE)
_lons = [c[0] for c in ROI_COORDS]
_lats = [c[1] for c in ROI_COORDS]
CX, CY = (min(_lons) + max(_lons)) / 2, (min(_lats) + max(_lats)) / 2
HW, HH = (max(_lons) - min(_lons)) / 2, (max(_lats) - min(_lats)) / 2

BBOXES = {f: (CX - HW * f, CY - HH * f, CX + HW * f, CY + HH * f) for f in FACTORES}
CAJAS = {
    f: ee.Geometry.Rectangle(list(BBOXES[f]), proj="EPSG:4326", geodesic=False)
    for f in FACTORES
}

print(f"Centro del ROI: {CY:.6f}, {CX:.6f}\n")
for f in FACTORES:
    x0, y0, x1, y1 = BBOXES[f]
    ancho = (x1 - x0) * 111.32 * math.cos(math.radians(CY))
    alto = (y1 - y0) * 111.32
    print(f"  {f}x → {ancho:5.2f} km × {alto:5.2f} km   (área ≈ {ancho*alto:5.2f} km²)")

## 6. Clasificación espacial y distancias

La distancia se calcula con **haversine** (esférica) en vez de la aproximación plana anterior.
A escalas de 10–150 km la diferencia es de decenas de metros, pero es el número correcto para citar.

In [ ]:
def dist_haversine_km(lat, lon, lat0: float, lon0: float, R: float = 6371.0088) -> np.ndarray:
    """Distancia esférica desde cada punto hasta (lat0, lon0), en km. Vectorizada."""
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    dphi = np.radians(lat0 - lat)
    dlam = np.radians(lon0 - lon)
    a = (
        np.sin(dphi / 2) ** 2
        + np.cos(np.radians(lat)) * math.cos(math.radians(lat0)) * np.sin(dlam / 2) ** 2
    )
    return 2 * R * np.arcsin(np.sqrt(a))


def clasificar_por_caja(df: pd.DataFrame) -> pd.Series:
    """Nivel de la caja más pequeña que contiene el punto; <NA> si ninguna."""
    nivel = pd.Series(pd.NA, index=df.index, dtype="Int32")
    for f in sorted(FACTORES, reverse=True):  # de mayor a menor: gana la más pequeña
        x0, y0, x1, y1 = BBOXES[f]
        dentro = df["Longitud"].between(x0, x1) & df["Latitud"].between(y0, y1)
        nivel = nivel.mask(dentro, f)
    return nivel


estaciones = catalogo.copy()
estaciones["nivel_caja"] = clasificar_por_caja(estaciones)
estaciones["dist_km"] = dist_haversine_km(
    estaciones["Latitud"], estaciones["Longitud"], CY, CX
).round(2)
estaciones = estaciones.sort_values("dist_km", kind="stable").reset_index(drop=True)

n_dentro = int(estaciones["nivel_caja"].notna().sum())
print(f"Estaciones dentro de alguna caja: {n_dentro}")
print(f"Estación más cercana: {estaciones['dist_km'].iat[0]:.2f} km\n")

estaciones[["CodigoEstacion", "NombreEstacion", "Municipio", "nivel_caja", "dist_km", "n_obs"]].head(10)

### Sitios físicos únicos

`0014015080` y `0014015020` son el **mismo instrumento** en el aeropuerto Rafael Núñez,
registrado dos veces en el catálogo (el segundo con el sufijo `TX GPRS`, que indica la vía de
transmisión telemétrica, no una estación distinta). Contarlas como dos estaciones sería un error
que un revisor familiarizado con la red del IDEAM notaría.

In [ ]:
def colapsar_sitios(estaciones: pd.DataFrame, tol: int = TOL_SITIO) -> pd.DataFrame:
    """Colapsa entradas de catálogo que corresponden al mismo sitio físico."""
    e = estaciones.sort_values("dist_km", kind="stable").copy()
    e["_la"] = e["Latitud"].round(tol)
    e["_lo"] = e["Longitud"].round(tol)

    g = e.groupby(["_la", "_lo"])["CodigoEstacion"]
    e["n_entradas_catalogo"] = g.transform("size")
    e["codigos_del_sitio"] = g.transform(lambda s: [list(s)] * len(s))

    return (
        e.drop_duplicates(["_la", "_lo"], keep="first")
        .drop(columns=["_la", "_lo"])
        .reset_index(drop=True)
    )


sitios = colapsar_sitios(estaciones)
print(f"{len(estaciones)} entradas de catálogo → {len(sitios)} sitios físicos\n")

sitios[["CodigoEstacion", "NombreEstacion", "dist_km", "n_entradas_catalogo", "codigos_del_sitio"]].head(6)

## 7. Mapa

El mapa se construye desde cero dentro de una función. En la versión anterior las capas se
acumulaban entre ejecuciones, lo que producía marcadores duplicados superpuestos.

In [ ]:
import ipyleaflet
from ipywidgets import HTML


def construir_mapa(sitios_sel: pd.DataFrame, zoom: int = 12) -> geemap.Map:
    """Mapa con el ROI, las cajas anidadas y las estaciones seleccionadas."""
    m = geemap.Map(center=[CY, CX], zoom=zoom)
    m.add_basemap("SATELLITE")

    # ROI
    m.addLayer(
        ee.Image().paint(ee.FeatureCollection([ee.Feature(roi)]), 0, 3),
        {"palette": "00FFFF"}, "Laguna (ROI)",
    )

    # Cajas, de la mayor a la menor para que la roja quede encima
    for f in sorted(FACTORES, reverse=True):
        m.addLayer(
            ee.Image().paint(ee.FeatureCollection([ee.Feature(CAJAS[f])]), 0, 3),
            {"palette": COLORES_CAJA[f]}, f"Caja {f}x",
        )

    # Estaciones: línea al centro del ROI + etiqueta de distancia + marcador
    paleta = ["#FF00FF", "#00FFAA", "#FFAA00", "#AA66FF"]
    for i, (_, r) in enumerate(sitios_sel.iterrows()):
        lat_e, lon_e, color = r["Latitud"], r["Longitud"], paleta[i % len(paleta)]

        m.add(ipyleaflet.Polyline(
            locations=[(CY, CX), (lat_e, lon_e)],
            color=color, weight=3, opacity=0.9, fill=False,
        ))

        # Etiqueta a distinta fracción del trayecto para que no se encimen
        frac = 0.5 - 0.12 * i
        m.add(ipyleaflet.Marker(
            location=(CY + (lat_e - CY) * frac, CX + (lon_e - CX) * frac),
            draggable=False,
            icon=ipyleaflet.DivIcon(
                html=(
                    f'<div style="background:{color};color:#000;padding:3px 8px;'
                    f'border-radius:4px;font-weight:bold;font-size:13px;'
                    f'white-space:nowrap;border:1px solid #000;">'
                    f'{r["d_borde_km"]:.2f} km</div>'
                ),
                icon_size=[0, 0],
            ),
        ))

        m.add_marker(
            location=(lat_e, lon_e),
            popup=HTML(
                f"<b>{r['NombreEstacion']}</b><br>"
                f"Código: {r['CodigoEstacion']}<br>"
                f"{r['Municipio']}<br>"
                f"Al centro del ROI: {r['d_centro_km']:.2f} km<br>"
                f"Al borde del ROI: {r['d_borde_km']:.2f} km<br>"
                f"Observaciones: {r['n_obs']:,}"
            ),
        )

    m.add_legend(title="Cajas", legend_dict={f"{f}x": "#" + COLORES_CAJA[f] for f in FACTORES})

    # Encuadre que cubre ROI y estaciones
    lats = [CY] + sitios_sel["Latitud"].tolist()
    lons = [CX] + sitios_sel["Longitud"].tolist()
    m.center = ((min(lats) + max(lats)) / 2, (min(lons) + max(lons)) / 2)
    return m

## 8. Selección de estaciones y serie temporal

Se reportan dos distancias: al **centro** del ROI y a su **borde**. La segunda es la que
conviene citar cuando se pregunte "¿a qué distancia del área de estudio?", porque no depende
del tamaño del polígono.

In [ ]:
# Distancias geodésicas exactas vía GEE, solo para las candidatas seleccionadas
seleccion = sitios.head(N_ESTACIONES).copy()
p_centro = ee.Geometry.Point([CX, CY])

d_centro, d_borde = [], []
for _, r in seleccion.iterrows():
    p = ee.Geometry.Point([r["Longitud"], r["Latitud"]])
    d_centro.append(p_centro.distance(p, maxError=1).getInfo() / 1000)
    d_borde.append(roi.distance(p, maxError=1).getInfo() / 1000)

seleccion["d_centro_km"] = np.round(d_centro, 2)
seleccion["d_borde_km"] = np.round(d_borde, 2)

for _, r in seleccion.iterrows():
    print(f"{r['NombreEstacion'][:38]:<40} centro: {r['d_centro_km']:6.2f} km   "
          f"borde: {r['d_borde_km']:6.2f} km")

In [ ]:
Map = construir_mapa(seleccion)
Map

In [ ]:
# Serie cruda de las estaciones seleccionadas.
# Se incluyen TODOS los códigos de cada sitio físico (p. ej. las dos entradas del aeropuerto),
# y se les asigna el nombre canónico DEL SITIO para que no se dividan en series distintas.
mapa_sitio = (
    seleccion[["codigos_del_sitio", "NombreEstacion"]]
    .explode("codigos_del_sitio")
    .rename(columns={"codigos_del_sitio": "CodigoEstacion", "NombreEstacion": "Estacion"})
)
print("Códigos incluidos:", mapa_sitio["CodigoEstacion"].tolist())

serie = (
    df.merge(mapa_sitio, on="CodigoEstacion", how="inner")
    [["Estacion", "CodigoEstacion", "FechaObservacion", "ValorObservado"]]
    .sort_values(["Estacion", "FechaObservacion"], kind="stable")
    .reset_index(drop=True)
)

print(f"{len(serie):,} registros en {serie['Estacion'].nunique()} sitios")
serie.head()

## 9. Diagnóstico de cobertura y frecuencia de muestreo

Esta es la corrección más importante respecto a la versión anterior, que reportaba
completitudes de 819 % y 3100 %. El error tenía dos causas:

1. **Agrupar por `(código, nombre)`** fragmentaba cada estación en varias filas.
2. **Asumir muestreo horario.** Rafael Núñez registra a resolución sub-horaria en parte de su
   historia — las marcas de tipo `23:58:00` lo delatan. Dividir el número de observaciones entre
   las horas transcurridas da entonces porcentajes por encima de 100.

La solución es medir primero el **paso de muestreo real** (mediana de las diferencias entre
registros consecutivos) y usarlo como denominador. El `diff()` se calcula **dentro de cada
estación**: hacerlo sobre la tabla completa mezclaría el último registro de una con el primero
de la siguiente.

In [ ]:
def diagnosticar(serie: pd.DataFrame) -> pd.DataFrame:
    """Ventana temporal, paso de muestreo real y completitud por estación."""
    s = serie.sort_values(["Estacion", "FechaObservacion"], kind="stable").copy()

    # diff DENTRO de cada estación, no sobre la tabla entera
    s["_paso_s"] = s.groupby("Estacion")["FechaObservacion"].diff().dt.total_seconds()

    out = (
        s.groupby("Estacion")
        .agg(
            inicio=("FechaObservacion", "min"),
            fin=("FechaObservacion", "max"),
            n_obs=("ValorObservado", "size"),
            n_nulos=("ValorObservado", lambda x: int(x.isna().sum())),
            paso_mediano_s=("_paso_s", "median"),
            paso_minimo_s=("_paso_s", "min"),
            temp_media=("ValorObservado", "mean"),
            temp_min=("ValorObservado", "min"),
            temp_max=("ValorObservado", "max"),
        )
        .reset_index()
    )

    span_s = (out["fin"] - out["inicio"]).dt.total_seconds()
    out["paso_mediano_min"] = (out["paso_mediano_s"] / 60).round(1)
    out["paso_minimo_min"] = (out["paso_minimo_s"] / 60).round(2)
    out["anios"] = (span_s / (365.25 * 86400)).round(2)
    # completitud sobre el paso REAL, no sobre un supuesto horario
    out["pct_completitud"] = (out["n_obs"] / (span_s / out["paso_mediano_s"] + 1) * 100).round(1)
    out["temp_media"] = out["temp_media"].round(2)

    return out.drop(columns=["paso_mediano_s", "paso_minimo_s"]).sort_values("inicio")


ventana = diagnosticar(serie)
ventana[[
    "Estacion", "inicio", "fin", "anios", "n_obs",
    "paso_mediano_min", "paso_minimo_min", "pct_completitud",
    "temp_media", "temp_min", "temp_max",
]]

In [ ]:
# Solape temporal entre estaciones: ¿existe un período común para validación cruzada?
cobertura_anual = (
    serie.assign(anio=serie["FechaObservacion"].dt.year)
    .pivot_table(index="anio", columns="Estacion", values="ValorObservado",
                 aggfunc="size", fill_value=0)
)
cobertura_anual

### Serie horaria homogénea

Como las estaciones muestrean a distinta frecuencia, compararlas registro a registro daría
un peso desproporcionado a la de mayor resolución. La reducción a paso horario las pone en
la misma escala. `n_crudos` conserva cuántas observaciones originales entraron en cada hora,
que es la trazabilidad que hace falta para justificar el promedio.

In [ ]:
horaria = (
    serie.dropna(subset=["ValorObservado"])
    .groupby(["Estacion", pd.Grouper(key="FechaObservacion", freq="h")])["ValorObservado"]
    .agg(temp="mean", n_crudos="size")
    .reset_index()
)
horaria["temp"] = horaria["temp"].round(2)

print(f"{len(serie):,} registros crudos → {len(horaria):,} horas")

resumen_horario = (
    horaria.groupby("Estacion")
    .agg(
        inicio=("FechaObservacion", "min"),
        fin=("FechaObservacion", "max"),
        n_horas=("temp", "size"),
        min_por_hora=("n_crudos", "min"),
        mediana_por_hora=("n_crudos", "median"),
        max_por_hora=("n_crudos", "max"),
    )
    .reset_index()
)
resumen_horario["horas_teoricas"] = (
    (resumen_horario["fin"] - resumen_horario["inicio"]).dt.total_seconds() // 3600 + 1
).astype(int)
resumen_horario["pct_horas_con_dato"] = (
    resumen_horario["n_horas"] / resumen_horario["horas_teoricas"] * 100
).round(1)

resumen_horario[[
    "Estacion", "inicio", "fin", "n_horas", "pct_horas_con_dato",
    "min_por_hora", "mediana_por_hora", "max_por_hora",
]]

### Visualización

In [ ]:
PALETA = ["#D64545", "#2E8B8B", "#5B7FBD", "#C08A2E"]
nombres = horaria["Estacion"].drop_duplicates().tolist()

# Estadísticos por serie (sobre la serie horaria homogeneizada)
stats = (
    horaria.groupby("Estacion")["temp"]
    .agg(n="size", media="mean", mediana="median", std="std", minimo="min", maximo="max")
    .round(2)
    .reindex(nombres)
)
display(stats)

fig, axes = plt.subplots(len(nombres), 1, figsize=(15, 3.6 * len(nombres)),
                         sharex=True, sharey=True)
axes = np.atleast_1d(axes)

for ax, nombre, color in zip(axes, nombres, PALETA):
    sub = horaria[horaria["Estacion"] == nombre]
    s = stats.loc[nombre]

    ax.plot(sub["FechaObservacion"], sub["temp"], lw=0.4, alpha=0.75, color=color)

    # Bandas y líneas de referencia
    ax.axhspan(s["media"] - s["std"], s["media"] + s["std"],
               color=color, alpha=0.10, zorder=0)
    ax.axhline(s["media"], color="black", lw=1.2, ls="-", alpha=0.8)
    ax.axhline(s["mediana"], color="black", lw=1.0, ls="--", alpha=0.7)
    ax.axhline(s["minimo"], color="#666", lw=0.8, ls=":", alpha=0.8)
    ax.axhline(s["maximo"], color="#666", lw=0.8, ls=":", alpha=0.8)

    # Etiquetas numéricas al margen derecho
    for val, txt, peso in [
        (s["media"],   f'media {s["media"]:.2f}',   "bold"),
        (s["mediana"], f'mediana {s["mediana"]:.2f}', "normal"),
        (s["minimo"],  f'mín {s["minimo"]:.2f}',    "normal"),
        (s["maximo"],  f'máx {s["maximo"]:.2f}',    "normal"),
    ]:
        ax.annotate(txt, xy=(1.0, val), xycoords=("axes fraction", "data"),
                    xytext=(4, 0), textcoords="offset points",
                    va="center", ha="left", fontsize=8, fontweight=peso, color="#333")

    # Recuadro resumen
    caja = (f'n = {int(s["n"]):,} h\n'
            f'media = {s["media"]:.2f} °C\n'
            f'mediana = {s["mediana"]:.2f} °C\n'
            f'σ = {s["std"]:.2f} °C\n'
            f'rango = {s["minimo"]:.2f} – {s["maximo"]:.2f} °C')
    ax.text(0.012, 0.04, caja, transform=ax.transAxes, fontsize=8.5,
            va="bottom", ha="left", family="monospace",
            bbox=dict(boxstyle="round,pad=0.45", facecolor="white",
                      edgecolor=color, alpha=0.9, linewidth=1.2))

    ax.set_title(nombre, fontsize=10, loc="left", fontweight="bold")
    ax.set_ylabel("°C")
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Fecha")
fig.suptitle("Temperatura del aire a 2 m — estaciones más cercanas al ROI",
             fontsize=12, y=0.995)
plt.tight_layout(rect=[0, 0, 0.93, 0.98])   # margen derecho para las etiquetas
plt.show()

In [ ]:
# Ciclo diario medio: comparación entre estaciones sobre una base homogénea
ciclo = (
    horaria.assign(hora=horaria["FechaObservacion"].dt.hour)
    .groupby(["Estacion", "hora"])["temp"]
    .agg(
        media="mean",
        p10=lambda x: x.quantile(0.10),
        p90=lambda x: x.quantile(0.90),
    )
    .round(2)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 4.5))
for nombre, color in zip(nombres, PALETA):
    s = ciclo[ciclo["Estacion"] == nombre]
    ax.plot(s["hora"], s["media"], marker="o", ms=4, color=color, label=nombre[:34])
    ax.fill_between(s["hora"], s["p10"], s["p90"], color=color, alpha=0.12)

ax.set_xlabel("Hora local")
ax.set_ylabel("Temperatura (°C)")
ax.set_title("Ciclo diario medio (banda: p10–p90)")
ax.set_xticks(range(0, 24, 2))
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**AQUI FALTAN COSAS COMO LO QUE VIENE DESPUES DE "JUNTAR O HACER MERGE" ELIMINAR DATOS RAROS DE LAS 2 SERIES ETC... PERO PRIMERO VEAMOS COMO SE VE EN LAS DEMAS VARIABLES**

## 10. Limitaciones a declarar en el manuscrito

**Ausencia de estaciones en el área de estudio.** Ninguna estación del IDEAM cae dentro de las
cajas anidadas, ni siquiera en la de mayor tamaño (≈ 3.4 × 3.9 km). La estación más próxima está
a ~8.9 km del borde del ROI. Esto no es un defecto del método sino una característica de la
densidad de la red automática en el norte de Bolívar, y conviene reportarlo como resultado.

**Sesgo espacial de la red disponible.** Las dos estaciones más cercanas se ubican al
oeste-suroeste, hacia el casco urbano de Cartagena. No existe cobertura al norte ni al este del
ROI, de modo que cualquier gradiente en esa dirección queda sin muestrear.

**Representatividad.** Rafael Núñez es una estación aeroportuaria sobre superficie
pavimentada; su temperatura del aire no equivale a la de una lámina de agua peri-urbana. Sirve
para régimen sinóptico (estacionalidad, anomalías interanuales), no para microclima lagunar.

**Heterogeneidad de muestreo.** Las estaciones no comparten paso temporal, y una misma estación
cambia de frecuencia a lo largo de su historia. Todo análisis comparativo debe partir de la serie
horaria homogeneizada, no de los registros crudos.